# NB2 - Neutrino masses and $N_{\rm eff}$
Tonale school, *Multimessenger cosmology*: Lecture 3 of the working-group material.

1. Neutrinos are radiation, then matter.
2. Massive neutrinos suppress the matter power spectrum: check $\Delta P/P\simeq-8f_\nu$.
3. What the CMB sees: lensing.
4. What BAO see: at fixed $\theta_s$, a neutrino mass raises $\Omega_m$.
5. $N_{\rm eff}$ in the CMB, one compensation at a time: what is left once $\theta_s$ and $z_{\rm eq}$ are fixed.
6. **Open question: two universes, which one has heavier neutrinos and which one has more radiation?**

Without `classy` the notebook reads precomputed arrays in `data/nb2.npz`: the plots are the same, but you cannot
change the parameters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    D = dict(np.load("data/nb2.npz"))
except FileNotFoundError:
    D = {}
try:                                  # CLASS is used only if it is installed and passes a quick test
    from classy import Class
    _c = Class(); _c.set({"h": 0.6736, "omega_b": 0.02237, "omega_cdm": 0.12, "YHe": 0.2454}); _c.compute()
    HAVE_CLASS = abs(_c.Neff() - 3.044) < 0.05          # some installations of classy return N_eff = 0
except Exception:
    HAVE_CLASS = False
if not HAVE_CLASS:
    print("classy is missing or fails a quick test (N_eff should be 3.044): "
          "using the precomputed arrays of data/nb2.npz")

def get(key, compute):
    "Run CLASS, or read the precomputed array with the same key."
    return np.array(compute()) if HAVE_CLASS else D[key]

plt.rcParams.update({"figure.figsize": (7, 4.5), "font.size": 13})
K = np.logspace(-3.2, 0.8, 100)       # 1/Mpc: physical wavenumbers, the same for every model
ELL = np.arange(2, 2501)

## 0. The models
`cosmo(sum_mnu, neff, omega_cdm, h)` is one CLASS run with three neutrinos of equal mass (massless if `sum_mnu = 0`)
and `neff` relativistic species in total. `find_h` tunes $h$ until the angular size of the sound horizon $\theta_s$
has the value measured by the CMB: this is what "fixing the CMB" means in practice.

In [ ]:
COMMON = {"omega_b": 0.02237, "A_s": 2.1e-9, "n_s": 0.9649, "tau_reio": 0.0544}
H_REF, OMC, OMB = 0.6736, 0.1200, 0.02237
THETA_S = 1.04037                     # 100 theta_s of the reference model (massless neutrinos, h = H_REF)

def cosmo(sum_mnu=0.0, neff=3.044, omega_cdm=OMC, h=H_REF, output="mPk", **extra):
    p = dict(COMMON, h=h, omega_cdm=omega_cdm, output=output, **extra)
    if sum_mnu > 0:
        p.update({"N_ncdm": 1, "deg_ncdm": 3, "m_ncdm": sum_mnu / 3, "N_ur": neff - 3.044 + 0.00441})
    else:
        p["N_ur"] = neff
    if "mPk" in output:
        p.update({"P_k_max_h/Mpc": 12.0, "z_max_pk": 3.0})
    if "Cl" in output:
        p.update({"lensing": "yes", "l_max_scalars": 2700})
    c = Class(); c.set(p); c.compute()
    return c

def find_h(**kw):
    "h such that 100 theta_s = THETA_S (secant method on cheap runs without perturbations)."
    h0, h1 = 0.64, 0.70
    f0 = cosmo(h=h0, output="", **kw).theta_s_100() - THETA_S
    for _ in range(12):
        f1 = cosmo(h=h1, output="", **kw).theta_s_100() - THETA_S
        if abs(f1) < 2e-6:
            break
        h0, h1, f0 = h1, h1 - f1 * (h1 - h0) / (f1 - f0), f1
    return h1

def pk(c, z=0.0):
    "Linear P(k) in Mpc^3 on the grid K."
    return np.array([c.pk_lin(k, z) for k in K])

def dl_tt(c):
    cl = c.lensed_cl(2500)
    return ELL * (ELL + 1) * cl["tt"][2:] / (2 * np.pi) * (2.7255e6)**2, cl["pp"][2:]

if HAVE_CLASS:
    print("100 theta_s of the reference model: %.5f" % cosmo(output="").theta_s_100())

## 1. Radiation, then matter
The neutrino density relative to the photon density. As long as neutrinos are relativistic the ratio is constant,
$3\times\frac78\left(\frac4{11}\right)^{4/3}\simeq0.69$; once their momentum drops below their mass it grows like $a$.

**Look at**: read the scale factor of the transition for each mass and compare with $1+z_{\rm nr}\simeq1890\,(m_\nu/{\rm eV})$
($m_\nu$ is the mass of one neutrino, a third of the sum). Are neutrinos with $\sum m_\nu=0.06$ eV relativistic at
recombination? Today?

In [ ]:
def nu_history(sum_mnu):
    bg = cosmo(sum_mnu, output="").get_background()
    a = np.logspace(-6, 0, 300)
    return a, np.interp(a, 1 / (1 + bg["z"]), bg["(.)rho_ncdm[0]"] / bg["(.)rho_g"])

for m in (0.06, 0.3, 1.2):
    a, r = get("history_%g" % m, lambda: nu_history(m))
    plt.loglog(a, r, label=r"$\sum m_\nu=%g$ eV" % m)
    plt.axvline(1 / (1890 * m / 3), color="grey", ls=":", lw=1)
plt.axvline(1 / 1090, color="k", lw=0.8); plt.text(1 / 1000, 30, "recombination", rotation=90, va="center")
plt.xlabel("a"); plt.ylabel(r"$\rho_\nu/\rho_\gamma$"); plt.legend();

## 2. The suppression of the matter power spectrum
Ratio to a universe with massless neutrinos and the **same** $\omega_m$ (the cold dark matter density is lowered by
$\omega_\nu=\sum m_\nu/93.14$ eV). Dashed: $-8f_\nu$ with $f_\nu=\omega_\nu/\omega_m$.

**Look at**: on which scales is there no suppression at all, and why? Is $-8f_\nu$ a good estimate of the plateau?
Why is the suppression a little weaker at $z=3$?

In [ ]:
def pk_ratio(sum_mnu, z):
    return pk(cosmo(sum_mnu, omega_cdm=OMC - sum_mnu / 93.14), z) / pk(cosmo(), z)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.3), sharey=True)
for i, m in enumerate((0.1, 0.2, 0.4)):
    for a, z in zip(ax, (0.0, 3.0)):
        a.semilogx(K, get("pk_%g_z%g" % (m, z), lambda: pk_ratio(m, z)), color="C%d" % i,
                   label=r"$\sum m_\nu=%g$ eV" % m)
        a.axhline(1 - 8 * m / 93.14 / (OMC + OMB), color="C%d" % i, ls="--", lw=1)
for a, z in zip(ax, (0.0, 3.0)):
    a.set_title("z = %g" % z); a.set_xlabel("k [1/Mpc]")
ax[0].set_ylabel(r"$P/P_{m_\nu=0}$"); ax[0].legend();

## 3. What the CMB sees: mostly lensing
Now the CMB is "fixed": same $\omega_b$, $\omega_c$ and $\theta_s$ (the code retunes $h$). Left: the lensing
potential spectrum. Right: the temperature spectrum.

**Look at**: which of the two is more sensitive to the mass? Why are the primary anisotropies almost unchanged for
$\sum m_\nu<1$ eV? What are the small wiggles in the temperature ratio?

In [ ]:
def cmb_ratio(sum_mnu):
    ref = dl_tt(cosmo(h=find_h(), output="tCl,pCl,lCl"))
    new = dl_tt(cosmo(sum_mnu, h=find_h(sum_mnu=sum_mnu), output="tCl,pCl,lCl"))
    return new[0] / ref[0], new[1] / ref[1]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
for m in (0.1, 0.3, 0.6):
    tt, pp = get("cmb_%g" % m, lambda: cmb_ratio(m))
    ax[0].semilogx(ELL, pp, label=r"$\sum m_\nu=%g$ eV" % m); ax[1].plot(ELL, tt)
ax[0].set_xlabel("L"); ax[0].set_ylabel(r"$C_L^{\phi\phi}$ / massless"); ax[0].legend()
ax[1].set_xlabel(r"$\ell$"); ax[1].set_ylabel(r"$C_\ell^{TT}$ / massless"); ax[1].set_ylim(0.97, 1.03);

## 4. What BAO see: the mass raises $\Omega_m$
Same "fixed CMB". Neutrinos were radiation at recombination, so they do not change the sound horizon; today they
are matter. The table gives $h$, $\Omega_m$ and the BAO observable $D_M(z=0.5)/r_d$.

**Look at**: how much does $\Omega_m$ move per 0.1 eV? Compare with the error bar of BAO surveys on $\Omega_m$
(about 0.009). In which direction would a measured $\Omega_m$ *lower* than the CMB prediction push the mass?

In [ ]:
def geometry(sum_mnu):
    c = cosmo(sum_mnu, h=find_h(sum_mnu=sum_mnu), output="")
    return c.h(), c.Omega_m(), 1.5 * c.angular_distance(0.5) / c.rs_drag()

masses = [0.0, 0.1, 0.2, 0.3]
tab = np.array([get("geometry_%g" % m, lambda: geometry(m)) for m in masses])
print("sum m_nu [eV]      h     Omega_m   D_M(0.5)/r_d")
for m, (h, om, dm) in zip(masses, tab):
    print("    %.1f        %.4f   %.4f     %.3f" % (m, h, om, dm))
print("per 0.1 eV: Delta h = %+.4f, Delta Omega_m = %+.4f" % ((tab[1, 0] - tab[0, 0]), (tab[1, 1] - tab[0, 1])))
plt.plot(masses, tab[:, 1], "o-")
plt.xlabel(r"$\sum m_\nu$ [eV]"); plt.ylabel(r"$\Omega_m$ at fixed $\theta_s$, $\omega_b$, $\omega_c$");

## 5. $N_{\rm eff}$: more radiation
### The background
The counterpart of the plot of section 1: a mass adds matter at late times, $N_{\rm eff}$ adds radiation at early times.
Left: the density of neutrinos and of the extra radiation, relative to photons, for several $N_{\rm eff}$ and
$\sum m_\nu=0.06$ eV, all the other parameters fixed. Right: the expansion rate relative to the reference model.

**Look at**: the radiation density is $\rho_r=\rho_\gamma\,(1+0.2271\,N_{\rm eff})$: by how much does one more species raise it?
Why do all the curves meet at late times? How does equality move (printed below), and how would you put it back? When is the expansion faster, and by how much? Which two
lengths of the CMB does that shorten?

In [ ]:
def neff_history(neff, sum_mnu=0.06):
    "Neutrinos + extra radiation over photons, radiation over matter, H [1/Mpc], as functions of the scale factor."
    bg = cosmo(sum_mnu, neff=neff, output="").get_background()
    a, x = np.logspace(-6, 0, 300), 1 / (1 + bg["z"])
    rel = 3 * bg["(.)p_ncdm[0]"]                                    # the relativistic part of the neutrino density
    rad, mat = bg["(.)rho_g"] + bg["(.)rho_ur"] + rel, bg["(.)rho_b"] + bg["(.)rho_cdm"] + bg["(.)rho_ncdm[0]"] - rel
    return (a, np.interp(a, x, (bg["(.)rho_ncdm[0]"] + bg["(.)rho_ur"]) / bg["(.)rho_g"]), np.interp(a, x, rad / mat),
            np.interp(a, x, bg["H [1/Mpc]"]))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
h0 = get("neffbg_3.044", lambda: neff_history(3.044))[3]
for i, n in enumerate((3.044, 3.5, 4.0, 5.0)):
    a, r_nu, r_eq, hub = get("neffbg_%g" % n, lambda: neff_history(n))
    a_eq = np.exp(np.interp(0.0, -np.log(r_eq), np.log(a)))         # where radiation = matter
    ax[0].loglog(a, r_nu, color="C%d" % i, label=r"$N_{\rm eff}=%g$" % n)
    ax[1].semilogx(a, hub / h0, color="C%d" % i)
    print("N_eff = %5g : plateau = %.3f, today = %.2f; 1 + z_eq = %4.0f; H / H_ref = %.3f early, %.3f at recombination"
          % (n, r_nu[0], r_nu[-1], 1 / a_eq, (hub / h0)[0], np.interp(1 / 1090, a, hub / h0)))
for b in ax:
    b.axvline(1 / 1090, color="k", lw=0.8); b.set_xlabel("a")
ax[0].text(1 / 1000, 8, "recombination", rotation=90, va="center"); ax[0].legend()
ax[0].set_ylabel(r"$(\rho_\nu+\rho_{\rm extra})/\rho_\gamma$"); ax[0].set_title(r"$\sum m_\nu=0.06$ eV: radiation, then the same mass")
ax[1].set_ylabel(r"$H/H_{\rm ref}$"); ax[1].set_title(r"expansion rate, relative to $N_{\rm eff}=3.044$");

### In the CMB, one compensation at a time
We add one species of radiation, $N_{\rm eff}=4$, then give the model more and more freedom to hide it:

1. nothing else changes;
2. $h$ is retuned, to keep the angular size of the sound horizon $\theta_s$;
3. $\omega_c$ is also raised, to keep the redshift of equality $z_{\rm eq}$;
4. the helium fraction $Y_p$ is also lowered, to keep the damping angle $\theta_d$. BBN does not allow this: the step
   is only here to isolate what is left.

**Look at**: in step 1, why do the peaks move, and why are the first ones higher? Which parameter repairs what in
steps 2 and 3, and by how much does it have to move (see the table)? The sound horizon scales as $1/H$ and the damping
length as $1/\sqrt H$: explain what remains after step 3. What are the oscillations left after step 4 (hint:
free-streaming particles travel faster than sound)?

In [ ]:
THETA_D = 0.32819                     # 100 theta_d of the reference model (photon diffusion length / distance)

def theta_d(c):
    d = c.get_current_derived_parameters(["rd_star", "ra_star"])
    return 100 * d["rd_star"] / d["ra_star"]

def find_yhe(**kw):
    "Helium fraction such that 100 theta_d = THETA_D, with theta_s kept fixed (secant method)."
    def f(y):
        return theta_d(cosmo(h=find_h(YHe=y, **kw), YHe=y, output="", compute_damping_scale="yes", **kw)) - THETA_D
    y0, y1 = 0.24, 0.20
    f0 = f(y0)
    for _ in range(10):
        f1 = f(y1)
        if abs(f1) < 2e-5:
            break
        y0, y1, f0 = y1, y1 - f1 * (y1 - y0) / (f1 - f0), f1
    return y1

def neff_step(neff, fix):
    "[h, omega_c, Y_p, z_eq, 100 theta_s, 100 theta_d] then the TT spectrum. fix: '', 's', 'se' or 'sed'."
    oc = (OMC + OMB) * (1 + 0.2271 * neff) / (1 + 0.2271 * 3.044) - OMB if "e" in fix else OMC      # same z_eq
    kw = dict(neff=neff, omega_cdm=oc)
    if "d" in fix:
        kw["YHe"] = find_yhe(**kw)                                                                   # same theta_d
    h = find_h(**kw) if "s" in fix else H_REF                                                        # same theta_s
    c = cosmo(h=h, output="tCl,pCl,lCl", compute_damping_scale="yes", **kw)
    d = c.get_current_derived_parameters(["YHe", "z_eq", "100*theta_s"])
    return np.concatenate([[h, oc, d["YHe"], d["z_eq"], d["100*theta_s"], theta_d(c)], dl_tt(c)[0]])

def neff_tt(neff):
    "TT spectrum with theta_s and z_eq fixed (step 3)."
    return neff_step(neff, "se")[6:]

NEFF = 4.0
STEPS = [("", "1. nothing fixed", r"1. $N_{\rm eff}=%g$, nothing else changed" % NEFF),
         ("s", "2. theta_s", r"2. + $h$ retuned: same $\theta_s$"),
         ("se", "3. theta_s, z_eq", r"3. + $\omega_c$ raised: same $z_{\rm eq}$"),
         ("sed", "4. theta_s, z_eq, theta_d", r"4. + $Y_p$ lowered: same $\theta_d$")]
ref = get("neff_3.044", lambda: neff_tt(3.044))
fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
print("fixed                         h     omega_c    Y_p    z_eq  100theta_s  100theta_d")
print("%-27s %.4f  %.4f   %.3f  %5.0f    %.4f     %.4f" % ("reference (N_eff = 3.044)", H_REF, OMC, 0.245, 3403, THETA_S, THETA_D))
for k, (fix, name, lab) in enumerate(STEPS):
    r = get("neffstep_%g_%s" % (NEFF, fix or "none"), lambda: neff_step(NEFF, fix))
    print("%-27s %.4f  %.4f   %.3f  %5.0f    %.4f     %.4f" % ((name,) + tuple(r[:6])))
    ax[k // 2].plot(ELL, r[6:] / ref, label=lab, color="C%d" % k)
ax[0].set_ylim(0.82, 1.32); ax[1].set_ylim(0.85, 1.03)
for a, loc in zip(ax, ("upper right", "lower left")):
    a.axhline(1, color="k", lw=0.8); a.set_xlabel(r"$\ell$"); a.legend(loc=loc)
ax[0].set_ylabel(r"$C_\ell^{TT}$ / reference ($N_{\rm eff}=3.044$)"); ax[1].set_title("steps 3 and 4 (note the scale)");

### The damping tail for several $N_{\rm eff}$
Step 3 again ($\theta_s$ and $z_{\rm eq}$ fixed), now for several values of $N_{\rm eff}$.

**Look at**: where does the spectrum change most, and how does the effect grow with $N_{\rm eff}$? Which precision at
$\ell\sim2000$ would detect $\Delta N_{\rm eff}=0.1$?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
for n in (2.5, 3.044, 3.5, 4.0):
    tt = get("neff_%g" % n, lambda: neff_tt(n))
    ax[0].plot(ELL, tt, label=r"$N_{\rm eff}=%g$" % n); ax[1].plot(ELL, tt / ref)
ax[0].set_yscale("log"); ax[0].set_xlabel(r"$\ell$"); ax[0].set_ylabel(r"$\ell(\ell+1)C_\ell^{TT}/2\pi$ [$\mu$K$^2$]")
ax[0].legend(); ax[1].set_xlabel(r"$\ell$"); ax[1].set_ylabel(r"ratio to $N_{\rm eff}=3.044$");

### The matter power spectrum for several $N_{\rm eff}$
The same models ($\theta_s$ and $z_{\rm eq}$ fixed, so $h$ and $\omega_c$ move with $N_{\rm eff}$), now in $P(k)$ today.

**Look at**: which scales gain power and which lose, and where is the pivot? Why does $P(k)$ change at all if equality
is fixed (which densities are *not* the same)? Compare the shape with the step of a neutrino mass (section 2): could
you confuse the two?

In [ ]:
def neff_pk(neff, z=0.0):
    "P(k) on the grid K with theta_s and z_eq fixed (step 3)."
    oc = (OMC + OMB) * (1 + 0.2271 * neff) / (1 + 0.2271 * 3.044) - OMB
    return pk(cosmo(neff=neff, omega_cdm=oc, h=find_h(neff=neff, omega_cdm=oc)), z)

ref_pk = get("neffpk_3.044", lambda: neff_pk(3.044))
for i, n in enumerate((2.5, 3.044, 3.5, 4.0)):
    r = get("neffpk_%g" % n, lambda: neff_pk(n)) / ref_pk
    plt.semilogx(K, r, color="C%d" % i, label=r"$N_{\rm eff}=%g$" % n)
    print("N_eff = %5g : P / P_ref = %.3f, %.3f, %.3f, %.3f at k = 0.01, 0.1, 1, 5 /Mpc" % ((n,) + tuple(np.interp([0.01, 0.1, 1, 5], K, r))))
plt.axhline(1, color="k", lw=0.8); plt.xlabel("k [1/Mpc]"); plt.ylabel(r"$P(k)$ / reference ($N_{\rm eff}=3.044$), z = 0"); plt.legend();

## 6. Open question: which universe is which?
Two universes, **X** and **Y**, with the same $\theta_s$ and $\omega_b$ as ours. In one, the neutrinos are heavier.
In the other there is extra radiation. You are given **measurements**, with their error bars, of the temperature spectrum (in bins of
$\Delta\ell=60$) and of the matter power spectrum of each, relative to the reference model. `chi2(u, tt_ratio, pk_ratio_)`
compares a model with the data of universe `u`.

1. Which is which? Use both observables.
2. Estimate $\sum m_\nu$ in one and $N_{\rm eff}$ in the other, then check with `neff_tt` and `neff_pk`, or with
   `cosmo(sum_mnu, h=find_h(sum_mnu=...))` for a mass. Which values does the $\chi^2$ allow?
3. Could a single observable have told them apart? Which combination of messengers measures the mass best?

In [ ]:
def binned(tt_ratio):
    "A spectrum ratio given on ELL, averaged in the multipole bins of the data."
    edges = np.append(D["myst_ell"] - 30, D["myst_ell"][-1] + 30)
    return np.array([tt_ratio[(ELL >= lo) & (ELL < hi)].mean() for lo, hi in zip(edges[:-1], edges[1:])])

def chi2(u, tt_ratio=None, pk_ratio_=None):
    "chi2 of a model (ratios to the reference: TT on ELL, P(k) on K) against the data of universe u."
    c = 0.0
    if tt_ratio is not None:
        c += np.sum(((binned(tt_ratio) - D["myst_%s_tt" % u]) / D["myst_%s_tt_err" % u])**2)
    if pk_ratio_ is not None:
        c += np.sum(((np.interp(D["myst_k"], K, pk_ratio_) - D["myst_%s_pk" % u]) / D["myst_%s_pk_err" % u])**2)
    return c

fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
for u, c in zip("XY", ("C0", "C3")):
    ax[0].errorbar(D["myst_ell"], D["myst_%s_tt" % u], D["myst_%s_tt_err" % u], fmt=".", color=c, label="universe " + u)
    ax[1].errorbar(D["myst_k"], D["myst_%s_pk" % u], D["myst_%s_pk_err" % u], fmt="o", ms=4, color=c, label="universe " + u)
# overlay your models here, for example (it is not the answer):
# ax[1].plot(K, neff_pk(3.3) / ref_pk); print(chi2("X", pk_ratio_=neff_pk(3.3) / ref_pk))
ax[1].set_xscale("log")
for a in ax:
    a.axhline(1, color="k", lw=0.8)
ax[0].set_xlabel(r"$\ell$"); ax[0].set_ylabel(r"$C_\ell^{TT}$ / reference"); ax[0].legend()
ax[1].set_xlabel("k [1/Mpc]"); ax[1].set_ylabel(r"$P(k)$ / reference, z = 0")
print("%d + %d data points per universe" % (len(D["myst_ell"]), len(D["myst_k"])))